In [1]:
import os
import time
import random
import heapq
import glob
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score
from skimage.segmentation import mark_boundaries
from lime import lime_image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.models as models

# ==============================================================================
# 🎛️ MASTER NOTEBOOK CONFIGURATION
# ==============================================================================
DATASET_NAME = "fashionmnist"       # Options: cifar10, fashionmnist, eurosat, isic
STRATEGY = "tuned"          # Options: baseline, augmented, tuned, augmented_tuned

EXPERIMENT_NAME = f"{DATASET_NAME}_{STRATEGY}"
SAVE_DIR = f"{EXPERIMENT_NAME}_models"
DATA_DIR = f"./data_{DATASET_NAME}"
DATASET_STATS = {
    "cifar10":      {"mean": (0.4914, 0.4822, 0.4465), "std": (0.2470, 0.2435, 0.2616), "in_channels": 3},
    "fashionmnist": {"mean": (0.2860,), "std": (0.3530,), "in_channels": 1}, # True grayscale stats
    "eurosat":      {"mean": (0.3444, 0.3803, 0.4078), "std": (0.2037, 0.1366, 0.1148), "in_channels": 3},
    "isic":         {"mean": (0.7630, 0.5456, 0.5700), "std": (0.1409, 0.1526, 0.1699), "in_channels": 3} 
}

DATASET_MEAN = DATASET_STATS[DATASET_NAME]["mean"]
DATASET_STD = DATASET_STATS[DATASET_NAME]["std"]
IN_CHANNELS = DATASET_STATS[DATASET_NAME]["in_channels"]

print(f"📊 Using Normalization Stats -> Mean: {DATASET_MEAN} | Std: {DATASET_STD}")

# Create the dedicated directories immediately
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("="*80)
print(f"🚀 INITIALIZED NOTEBOOK FOR: [{EXPERIMENT_NAME.upper()}]")
print(f"📁 Dataset Directory: {DATA_DIR}/")
print(f"📁 Artifacts Directory: ./{SAVE_DIR}/")
print("="*80)

# Hardware device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Global reproducibility setup
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

import sys
import warnings

# 1. Kill all standard warnings (Deprecation, UserWarnings, etc.)
warnings.filterwarnings("ignore")

# 2. Kill all background garbage collection "Exception ignored in..." spam
sys.unraisablehook = lambda unraisable: None

# 3. Kill ALL fatal crash tracebacks 
sys.excepthook = lambda exctype, value, traceback: None

📊 Using Normalization Stats -> Mean: (0.286,) | Std: (0.353,)
🚀 INITIALIZED NOTEBOOK FOR: [FASHIONMNIST_TUNED]
📁 Dataset Directory: ./data_fashionmnist/
📁 Artifacts Directory: ./fashionmnist_tuned_models/
Using device: cuda


In [2]:
# ==============================================================================
# 2. UNIVERSAL DATA PREPROCESSING & LOADER
# ==============================================================================
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset

# --- A. DYNAMIC TRANSFORM PIPELINE ---
if STRATEGY in ["augmented", "augmented_tuned"]:
    if IN_CHANNELS == 1:
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2), # 1-channel safe
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0),
            transforms.Normalize(DATASET_MEAN, DATASET_STD),
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandAugment(num_ops=2, magnitude=9),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0),
            transforms.Normalize(DATASET_MEAN, DATASET_STD),
        ])
    print("🧬 Loaded AUGMENTED Transform Pipeline")
else:
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(DATASET_MEAN, DATASET_STD),
    ])
    print("🧼 Loaded CLEAN Baseline Transform Pipeline")

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(DATASET_MEAN, DATASET_STD),
])

# --- B. CUSTOM ISIC LOADER ---
class ISIC2019Dataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.img_dir, self.transform = img_dir, transform
        self.df = pd.read_csv(csv_file)
        self.classes = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']
        class_names = self.df[self.classes].idxmax(axis=1)
        self.targets = [self.classes.index(cls) for cls in class_names]
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df.iloc[idx]['image'] + '.jpg')
        image = Image.open(img_path).convert("RGB")
        if self.transform: image = self.transform(image)
        return image, self.targets[idx]

# --- C. DATASET ROUTER ---
if DATASET_NAME == "cifar10":
    train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
    test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

elif DATASET_NAME == "fashionmnist":
    train_dataset = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=train_transform)
    test_dataset  = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=test_transform)

elif DATASET_NAME in ["eurosat", "isic"]:
    if DATASET_NAME == "eurosat":
        full_train = torchvision.datasets.EuroSAT(root=DATA_DIR, download=True, transform=train_transform)
        full_test  = torchvision.datasets.EuroSAT(root=DATA_DIR, download=True, transform=test_transform)
    else:
        csv_path, img_dir = "./ISIC_2019_Training_GroundTruth.csv", "./ISIC_2019_Training_Input"
        if not os.path.exists(csv_path): raise FileNotFoundError(f"⚠️ ISIC CSV missing: {csv_path}")
        full_train = ISIC2019Dataset(csv_file=csv_path, img_dir=img_dir, transform=train_transform)
        full_test  = ISIC2019Dataset(csv_file=csv_path, img_dir=img_dir, transform=test_transform)
    
    indices = np.random.RandomState(42).permutation(len(full_train))
    t_idx, v_idx = indices[:int(0.8 * len(full_train))], indices[int(0.8 * len(full_train)):]
    
    train_dataset, test_dataset = Subset(full_train, t_idx), Subset(full_test, v_idx)
    
    train_dataset.targets, train_dataset.classes = [full_train.targets[i] for i in t_idx], full_train.classes
    test_dataset.targets, test_dataset.classes = [full_test.targets[i] for i in v_idx], full_test.classes

else:
    raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}")

# --- D. DATALOADER INITIALIZATION ---
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

sample_imgs, sample_labels = next(iter(train_loader))
print(f"📦 [{DATASET_NAME.upper()}] Batch shape initialized: {sample_imgs.shape}")

🧼 Loaded CLEAN Baseline Transform Pipeline
📦 [FASHIONMNIST] Batch shape initialized: torch.Size([32, 1, 224, 224])


In [3]:
import torchvision.models as models
import torch.nn as nn
import torch
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
import time

# ==========================================
# Architectures
# ==========================================
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10, in_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            # 🔥 DYNAMIC CHANNELS: Accepts 1 (Grayscale) or 3 (RGB)
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

def build_model(name, num_classes=10, in_channels=3):
    if name == "Baseline_CNN":
        return BaselineCNN(num_classes=num_classes, in_channels=in_channels)
        
    elif name == "ResNet18":
        m = models.resnet18(weights=None)
        # 🔥 SURGICAL FIX: Replace first Conv layer if dataset is grayscale
        if in_channels != 3:
            m.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
        
    elif name == "MobileNetV2":
        m = models.mobilenet_v2(weights=None)
        # 🔥 SURGICAL FIX: Replace first Conv layer if dataset is grayscale
        if in_channels != 3:
            m.features[0][0] = nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ==========================================
# Step Logic
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

In [ ]:
import os
import glob
import copy
import time
import json
import optuna
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader
from tqdm.auto import tqdm

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_val_acc = 0.0
        self.best_train_acc = 0.0
        self.best_model_weights = None
        self.early_stop = False

    def step(self, val_loss, val_acc, train_acc, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_val_acc = val_acc
            self.best_train_acc = train_acc
            self.best_model_weights = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def restore_best_weights(self, model):
        if self.best_model_weights is not None:
            model.load_state_dict(self.best_model_weights)

# ==============================================================================
# 🎛️ TRAINING & SEARCH CONFIGURATION
# ==============================================================================
MAX_EPOCHS = 50      # 50 is standard for production training
PATIENCE = 5
BATCH_SIZE = 32

OPTUNA_TRIALS = 10   # Standard number of search attempts
OPTUNA_EPOCHS = 5    # Enough trajectory for Bayesian prediction

# 🔥 DYNAMIC CLASS COUNT 
NUM_CLASSES = len(train_dataset.classes)
print(f"Dynamically detected {NUM_CLASSES} classes in the training dataset.")

targets = np.array(train_dataset.targets)
criterion = nn.CrossEntropyLoss()
models_to_benchmark = ["Baseline_CNN", "ResNet18", "MobileNetV2"]

# ==============================================================================
# 🗂️ THE MASTER DATA SPLIT (Single 80/20 Split replacing K-Fold)
# ==============================================================================
# We do this once globally so Optuna and the final training use the exact same holdout
train_idx, val_idx = train_test_split(
    np.arange(len(targets)), test_size=0.2, stratify=targets, random_state=42
)

master_train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
master_val_loader = DataLoader(Subset(train_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ==============================================================================
# 🔍 OPTUNA HYPERPARAMETER SEARCH (Runs conditionally based on STRATEGY)
# ==============================================================================
tuned_params = {}
params_path = f"{SAVE_DIR}/tuned_params.json"

if STRATEGY in ["tuned", "augmented_tuned"]:
    # 1. Load partial progress if the file exists
    if os.path.exists(params_path):
        with open(params_path, 'r') as f:
            tuned_params = json.load(f)
        print(f"\n📥 Loaded existing tuning progress for: {list(tuned_params.keys())}")
    else:
        print("\n" + "="*80)
        print("🔍 INITIATING STANDARD OPTUNA HYPERPARAMETER SEARCH")
        print("="*80)

    # Fast 80/20 split from the training dataset just for searching
    t_idx, v_idx = train_test_split(np.arange(len(targets)), test_size=0.2, stratify=targets, random_state=42)
    
    search_train_loader = DataLoader(Subset(train_dataset, t_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    search_val_loader = DataLoader(Subset(train_dataset, v_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    for m_name in models_to_benchmark:
        # 2. THE CHECK: Skip this specific model if it already finished and saved
        if m_name in tuned_params:
            print(f"\n>>> [SKIP] {m_name} already tuned: {tuned_params[m_name]}")
            continue
            
        print(f"\n>>> Tuning {m_name}...")
        
        def objective(trial):
            lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
            wd = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
            
            model = build_model(m_name, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
            
            for epoch in range(1, OPTUNA_EPOCHS + 1):
                model.train()
                pbar_opt = tqdm(search_train_loader, desc=f"Trial {trial.number} | Ep {epoch}/{OPTUNA_EPOCHS}", leave=False)
                
                for images, labels in pbar_opt:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    pbar_opt.set_postfix({"loss": f"{loss.item():.4f}"})
                    
                val_loss, val_acc = evaluate_epoch(model, search_val_loader, criterion, device)
                
                trial.report(val_acc, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
                    
            return val_acc
        
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
        study.optimize(objective, n_trials=OPTUNA_TRIALS) 
        
        print(f" -> ✅ Best params for {m_name}: {study.best_params}")
        tuned_params[m_name] = study.best_params
        
        # 3. THE FIX: SAVE IMMEDIATELY AFTER EVERY MODEL (Notice the indentation)
        with open(params_path, 'w') as f:
            json.dump(tuned_params, f, indent=4)

else:
    # Safe Fallback for Baseline / Augmented variants
    for m_name in models_to_benchmark:
        tuned_params[m_name] = {"lr": 1e-3, "weight_decay": 1e-2}

# ==============================================================================
# 🏋️ THE FINAL TRAINING BENCHMARK (Single Split)
# ==============================================================================
for m_idx, model_name in enumerate(models_to_benchmark, 1):
    print("\n" + "="*80)
    print(f"🚀 ACTIVE BENCHMARK [{m_idx}/{len(models_to_benchmark)}]: {model_name}")
    print("="*80)
    
    # We leave "_fold1_" in the string purely so Snippet 8's regex scanner still works
    search_pattern = f"{SAVE_DIR}/{model_name}_fold1_val*_train*.pt"
    existing_files = glob.glob(search_pattern)
    
    if existing_files:
        print(f"\n>>> [SKIP] Found existing artifact for {model_name}: {existing_files[0]}")
        continue
        
    print(f"\n>>> Model: [{model_name}] | Running Final Training Phase...")
    
    model = build_model(model_name, num_classes=NUM_CLASSES, in_channels=IN_CHANNELS).to(device)
    
    # 🔥 APPLY TUNED PARAMETERS HERE
    lr = tuned_params[model_name]["lr"]
    wd = tuned_params[model_name]["weight_decay"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    
    early_stopping = EarlyStopping(patience=PATIENCE)
    
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        pbar = tqdm(master_train_loader, desc=f"[{model_name}] | Ep {epoch:02d}/{MAX_EPOCHS}", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix({"loss": f"{running_loss / total:.4f}"})
            
        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate_epoch(model, master_val_loader, criterion, device)
        
        print(f"[{model_name}] Ep {epoch:02d} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} (Acc: {val_acc:.2f}%)")
        
        early_stopping.step(val_loss, val_acc, train_acc, model)
        if early_stopping.early_stop:
            print(f" 🛑 Early stopping at epoch {epoch}.")
            break
            
    early_stopping.restore_best_weights(model)
    
    b_val = early_stopping.best_val_acc
    b_train = early_stopping.best_train_acc
    
    save_path = f"{SAVE_DIR}/{model_name}_fold1_val{b_val:.2f}_train{b_train:.2f}.pt"
    
    torch.save({
        'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'val_acc': b_val,
        'classes': getattr(train_dataset, 'classes', None), 
        'input_size': (IN_CHANNELS, 224, 224),
    }, save_path)
    print(f" -> ✅ Saved Artifact: {save_path}")

Dynamically detected 10 classes in the training dataset.

🔍 INITIATING STANDARD OPTUNA HYPERPARAMETER SEARCH

>>> Tuning Baseline_CNN...


Trial 0 | Ep 1/5:   0%|          | 0/1500 [00:08<?, ?it/s]

Trial 0 | Ep 2/5:   0%|          | 0/1500 [00:05<?, ?it/s]

Trial 0 | Ep 3/5:   0%|          | 0/1500 [00:05<?, ?it/s]

Trial 0 | Ep 4/5:   0%|          | 0/1500 [00:05<?, ?it/s]

Trial 0 | Ep 5/5:   0%|          | 0/1500 [00:05<?, ?it/s]

... (139 outputs left out)

[Baseline_CNN] Ep 20 | Train Acc: 90.36% | Val Loss: 0.4568 (Acc: 84.28%)


[Baseline_CNN] | Ep 21/50:   0%|          | 0/1500 [00:05<?, ?it/s]

[Baseline_CNN] Ep 21 | Train Acc: 90.46% | Val Loss: 0.2881 (Acc: 90.08%)
 🛑 Early stopping at epoch 21.
 -> ✅ Saved Artifact: fashionmnist_tuned_models/Baseline_CNN_fold1_val90.60_train89.76.pt

🚀 ACTIVE BENCHMARK [2/3]: ResNet18

>>> Model: [ResNet18] | Running Final Training Phase...


[ResNet18] | Ep 01/50:   0%|          | 0/1500 [00:05<?, ?it/s]

In [ ]:
# ==============================================================================
# Module 3.1: The Perturbation Engine (Section 4.5.1 & 4.5.2)
# ==============================================================================
class PerturbationEngine:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
        self.normalize = transforms.Normalize(mean=mean, std=std)

    def gaussian_blur(self, x: torch.Tensor, intensity: float) -> torch.Tensor:
        if intensity <= 0.0: return x
        sigma = float(0.1 + intensity * 4.9)
        k_size = int(3 + int(intensity * 6) * 2)
        return TF.gaussian_blur(x, kernel_size=[k_size, k_size], sigma=[sigma, sigma])

    def gaussian_noise(self, x: torch.Tensor, intensity: float) -> torch.Tensor:
        if intensity <= 0.0: return x
        noise_std = intensity * 0.35
        noise = torch.randn_like(x) * noise_std
        return torch.clamp(x + noise, 0.0, 1.0)

    def brightness_shift(self, x: torch.Tensor, intensity: float) -> torch.Tensor:
        if intensity <= 0.0: return x
        shift = intensity * 0.6
        return torch.clamp(x + shift, 0.0, 1.0)

    def rotation(self, x: torch.Tensor, intensity: float) -> torch.Tensor:
        if intensity <= 0.0: return x
        angle = intensity * 90.0
        return TF.rotate(x, angle=angle)

    def contrast_exposure(self, x: torch.Tensor, intensity: float) -> torch.Tensor:
        if intensity <= 0.0: return x
        factor = float(1.0 + intensity * 2.0)
        return TF.adjust_contrast(x, contrast_factor=factor)

    def apply_raw_perturbation(self, x: torch.Tensor, p_type: str, intensity: float) -> torch.Tensor:
        """Returns perturbed image in unnormalized [0, 1] range."""
        x = x.clone()
        if p_type == "gaussian_blur":
            out = self.gaussian_blur(x, intensity)
        elif p_type == "gaussian_noise":
            out = self.gaussian_noise(x, intensity)
        elif p_type == "brightness":
            out = self.brightness_shift(x, intensity)
        elif p_type == "rotation":
            out = self.rotation(x, intensity)
        elif p_type == "contrast":
            out = self.contrast_exposure(x, intensity)
        elif p_type == "sensor_interference":  # Blur + Noise
            out = self.gaussian_blur(x, intensity * 0.8)
            out = self.gaussian_noise(out, intensity * 0.8)
        elif p_type == "environmental_obstruction":  # Blur + Brightness
            out = self.gaussian_blur(x, intensity * 0.7)
            out = self.brightness_shift(out, intensity * 0.7)
        elif p_type == "capture_instability":  # Rotation + Blur
            out = self.rotation(x, intensity * 0.8)
            out = self.gaussian_blur(out, intensity * 0.8)
        else:
            raise ValueError(f"Unknown perturbation: {p_type}")
        return out

    def apply_perturbation(self, x: torch.Tensor, p_type: str, intensity: float) -> torch.Tensor:
        """Returns perturbed image normalized for PyTorch model ingestion."""
        raw_perturbed = self.apply_raw_perturbation(x, p_type, intensity)
        return self.normalize(raw_perturbed)

# ==============================================================================
# Module 3.2: Adaptive Heuristic Sampling (Fragility Gradient)
# ==============================================================================
class AdaptiveSamplingEngine:
    def __init__(self, model, dataloader, engine: PerturbationEngine, device):
        self.model = model.to(device)
        self.model.eval()
        self.dataloader = dataloader
        self.engine = engine
        self.device = device

    @torch.no_grad()
    def evaluate_at_intensity(self, p_type: str, intensity: float, max_batches: int = None) -> float:
        correct, total = 0, 0
        for b_idx, (images, labels) in enumerate(self.dataloader):
            if max_batches and b_idx >= max_batches:
                break
            perturbed_images = self.engine.apply_perturbation(images, p_type, intensity).to(self.device)
            labels = labels.to(self.device)
            outputs = self.model(perturbed_images)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        return (correct / total) * 100.0

    def run_adaptive_sampling(self, p_type: str, max_evals: int = 9, max_batches: int = 15):
        eval_cache = {}
        acc_min = self.evaluate_at_intensity(p_type, 0.0, max_batches=max_batches)
        acc_max = self.evaluate_at_intensity(p_type, 1.0, max_batches=max_batches)
        eval_cache[0.0] = acc_min
        eval_cache[1.0] = acc_max

        heap = []
        def compute_slope(x1, x2):
            dx = x2 - x1
            dy = eval_cache[x2] - eval_cache[x1]
            return abs(dy / dx) if dx > 0 else 0.0

        heapq.heappush(heap, (-compute_slope(0.0, 1.0), 0.0, 1.0))

        while len(eval_cache) < max_evals and heap:
            neg_slope, x1, x2 = heapq.heappop(heap)
            x_mid = round((x1 + x2) / 2.0, 4)
            if x_mid in eval_cache or abs(x2 - x1) < 1e-3:
                continue

            acc_mid = self.evaluate_at_intensity(p_type, x_mid, max_batches=max_batches)
            eval_cache[x_mid] = acc_mid

            heapq.heappush(heap, (-compute_slope(x1, x_mid), x1, x_mid))
            heapq.heappush(heap, (-compute_slope(x_mid, x2), x_mid, x2))

        sorted_samples = sorted(eval_cache.items(), key=lambda x: x[0])
        intensities, accuracies = zip(*sorted_samples)
        
        # Determine Critical Failure Threshold (Intensity where steepest accuracy collapse occurred)
        steepest_grad = -1.0
        critical_threshold = 0.5
        for idx in range(len(intensities) - 1):
            dx = intensities[idx+1] - intensities[idx]
            dy = abs(accuracies[idx+1] - accuracies[idx])
            grad = dy / dx if dx > 0 else 0.0
            if grad > steepest_grad:
                steepest_grad = grad
                critical_threshold = (intensities[idx] + intensities[idx+1]) / 2.0

        return list(intensities), list(accuracies), critical_threshold

In [ ]:
class MetricEngine:
    """
    Module 4: Computes Accuracy, MCC, F1, Flip Rate, ECE, and NLL.
    """
    @staticmethod
    def calculate_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
        """Calculates Expected Calibration Error (ECE)."""
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        confidences = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)
        accuracies = (predictions == labels)

        ece = 0.0
        for i in range(n_bins):
            bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i+1]
            in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
            prop_in_bin = np.mean(in_bin)

            if prop_in_bin > 0:
                accuracy_in_bin = np.mean(accuracies[in_bin])
                avg_confidence_in_bin = np.mean(confidences[in_bin])
                ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
        return float(ece)

    @torch.no_grad()
    def evaluate_comprehensive_metrics(self, model, dataloader, engine: PerturbationEngine, 
                                       p_type: str, intensity: float, device, max_batches: int = 20):
        model.eval()
        
        all_clean_preds = []
        all_pert_preds = []
        all_labels = []
        all_pert_probs = []
        nll_losses = []

        nll_criterion = nn.CrossEntropyLoss()

        for b_idx, (images, labels) in enumerate(dataloader):
            if max_batches and b_idx >= max_batches:
                break
                
            labels_dev = labels.to(device)
            
            # Clean baseline inference
            clean_images = engine.normalize(images.clone()).to(device)
            clean_logits = model(clean_images)
            _, clean_preds = clean_logits.max(1)
            
            # Perturbed inference
            pert_images = engine.apply_perturbation(images, p_type, intensity).to(device)
            pert_logits = model(pert_images)
            pert_loss = nll_criterion(pert_logits, labels_dev)
            
            pert_probs = F.softmax(pert_logits, dim=1)
            _, pert_preds = pert_logits.max(1)

            # Store for batch-independent calculation
            all_clean_preds.extend(clean_preds.cpu().numpy())
            all_pert_preds.extend(pert_preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_pert_probs.extend(pert_probs.cpu().numpy())
            nll_losses.append(pert_loss.item())

        y_true = np.array(all_labels)
        y_pred = np.array(all_pert_preds)
        y_clean = np.array(all_clean_preds)
        probs = np.array(all_pert_probs)

        # 1. Classification Metrics
        acc = accuracy_score(y_true, y_pred) * 100.0
        mcc = matthews_corrcoef(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average='macro')

        # 2. Instability Metric: Flip Rate (Fraction of clean predictions that altered under noise)
        flip_rate = np.mean(y_clean != y_pred) * 100.0

        # 3. Calibration Metrics
        ece = self.calculate_ece(probs, y_true, n_bins=15)
        nll = float(np.mean(nll_losses))

        return {
            "Accuracy (%)": acc,
            "MCC": mcc,
            "Macro F1": macro_f1,
            "Flip Rate (%)": flip_rate,
            "ECE": ece,
            "NLL": nll
        }

In [ ]:
class LIMEExplainabilityEngine:
    """
    Module 5: Generates LIME superpixel explanations at critical failure thresholds.
    """
    def __init__(self, model, engine: PerturbationEngine, classes, device):
        self.model = model.to(device)
        self.model.eval()
        self.engine = engine
        self.classes = classes
        self.device = device
        self.explainer = lime_image.LimeImageExplainer()

    def _batch_predict(self, numpy_images: np.ndarray) -> np.ndarray:
        """Wrapper function for LIME: takes numpy images (N, H, W, 3) in [0, 1] and returns class probabilities."""
        self.model.eval()
        # Convert (N, H, W, C) -> (N, C, H, W)
        tensor_batch = torch.tensor(numpy_images, dtype=torch.float32).permute(0, 3, 1, 2)
        norm_batch = self.engine.normalize(tensor_batch).to(self.device)
        with torch.no_grad():
            logits = self.model(norm_batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()
        return probs

    def explain_critical_sample(self, raw_img_tensor: torch.Tensor, label_idx: int, 
                                p_type: str, critical_intensity: float, num_samples: int = 150):
        """
        Generates side-by-side LIME visualizations: Clean focus vs Perturbed focus at threshold.
        """
        # Ensure single image shape (3, H, W)
        if raw_img_tensor.dim() == 4:
            raw_img_tensor = raw_img_tensor.squeeze(0)

        # 1. Prepare Clean Image (H, W, 3) numpy in [0, 1]
        clean_np = raw_img_tensor.permute(1, 2, 0).cpu().numpy()
        
        # 2. Prepare Perturbed Image at Critical Threshold
        pert_tensor = self.engine.apply_raw_perturbation(raw_img_tensor.unsqueeze(0), p_type, critical_intensity).squeeze(0)
        pert_np = pert_tensor.permute(1, 2, 0).cpu().numpy()

        # Generate LIME explanations
        clean_explanation = self.explainer.explain_instance(
            clean_np.astype(np.double), 
            self._batch_predict, 
            top_labels=1, 
            hide_color=0, 
            num_samples=num_samples
        )
        
        pert_explanation = self.explainer.explain_instance(
            pert_np.astype(np.double), 
            self._batch_predict, 
            top_labels=1, 
            hide_color=0, 
            num_samples=num_samples
        )

        clean_top_pred = clean_explanation.top_labels[0]
        pert_top_pred = pert_explanation.top_labels[0]

        temp_clean, mask_clean = clean_explanation.get_image_and_mask(
            clean_top_pred, positive_only=True, num_features=5, hide_rest=False
        )
        temp_pert, mask_pert = pert_explanation.get_image_and_mask(
            pert_top_pred, positive_only=True, num_features=5, hide_rest=False
        )

        # Plot Comparative Focus Shift
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        
        axes[0].imshow(clean_np)
        axes[0].set_title(f"Clean Input\nTrue: {self.classes[label_idx]}", fontsize=10)
        axes[0].axis('off')

        axes[1].imshow(mark_boundaries(temp_clean, mask_clean))
        axes[1].set_title(f"Clean LIME Focus\nPred: {self.classes[clean_top_pred]}", fontsize=10)
        axes[1].axis('off')

        axes[2].imshow(pert_np)
        axes[2].set_title(f"Perturbed (I={critical_intensity:.2f})\nType: {p_type}", fontsize=10)
        axes[2].axis('off')

        axes[3].imshow(mark_boundaries(temp_pert, mask_pert))
        axes[3].set_title(f"Perturbed LIME Focus\nPred: {self.classes[pert_top_pred]}", fontsize=10)
        axes[3].axis('off')

        plt.suptitle(f"Spatial Attention Shift Analysis at Critical Threshold", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
import os
import glob
import re
import torch
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ==============================================================================
# 1. UNNORMALIZED HOLDOUT EVALUATION DATALOADER (Universal Router)
# ==============================================================================
raw_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print(f"\n📦 Loading RAW validation data for [{DATASET_NAME.upper()}]...")

if DATASET_NAME == "cifar10":
    raw_test_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=raw_test_transform)
elif DATASET_NAME == "fashionmnist":
    raw_test_dataset = torchvision.datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=raw_test_transform)
elif DATASET_NAME == "eurosat":
    full_raw = torchvision.datasets.EuroSAT(root=DATA_DIR, download=True, transform=raw_test_transform)
    indices = np.random.RandomState(42).permutation(len(full_raw))
    _, v_idx = indices[:int(0.8 * len(full_raw))], indices[int(0.8 * len(full_raw)):]
    raw_test_dataset = Subset(full_raw, v_idx)
    raw_test_dataset.classes = full_raw.classes
elif DATASET_NAME == "isic":
    csv_path = "./ISIC_2019_Training_GroundTruth.csv"
    img_dir = "./ISIC_2019_Training_Input"
    full_raw = ISIC2019Dataset(csv_file=csv_path, img_dir=img_dir, transform=raw_test_transform)
    indices = np.random.RandomState(42).permutation(len(full_raw))
    _, v_idx = indices[:int(0.8 * len(full_raw))], indices[int(0.8 * len(full_raw)):]
    raw_test_dataset = Subset(full_raw, v_idx)
    raw_test_dataset.classes = full_raw.classes

raw_test_loader = DataLoader(
    raw_test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

# 🔥 DYNAMIC LABEL EXTRACTION
classes = raw_test_dataset.classes 
print(f"Loaded Classes: {classes}")

# ==============================================================================
# 2. CHECKPOINT SCANNER
# ==============================================================================
trained_models = {}

print("\n" + "="*80)
print(f"📥 SCANNING FOR BEST CHECKPOINTS IN: {SAVE_DIR}/")
print("="*80)

for m_name in ["Baseline_CNN", "ResNet18", "MobileNetV2"]:
    search_pattern = f"{SAVE_DIR}/{m_name}_fold*_val*_train*.pt"
    available_files = glob.glob(search_pattern)
    
    if not available_files:
        print(f"⚠️ Warning: No checkpoints found for {m_name}")
        continue
        
    best_file, highest_val = None, -1.0
    
    for file_path in available_files:
        match = re.search(r'_val(\d+\.\d+)_', file_path)
        if match and float(match.group(1)) > highest_val:
            highest_val = float(match.group(1))
            best_file = file_path
                
    if best_file:
        print(f"[{m_name}] Loading top performer: {os.path.basename(best_file)}")
        ckpt = torch.load(best_file, map_location=device)
        m = build_model(m_name, num_classes=len(classes), in_channels=IN_CHANNELS)
        m.load_state_dict(ckpt['model_state_dict'])
        m = m.to(device)
        m.eval()
        trained_models[m_name] = m

# ==============================================================================
# 3. ENGINES & STRESS TEST EXECUTION
# ==============================================================================
pert_engine = PerturbationEngine(mean=DATASET_MEAN, std=DATASET_STD)
metric_engine = MetricEngine()

all_perturbations = [
    "gaussian_blur", "gaussian_noise", "brightness", "rotation", 
    "contrast", "sensor_interference", "environmental_obstruction", "capture_instability"
]

print("\n" + "="*95)
print("🚀 INITIATING MODULE 3 -> 4 -> 5 PIPELINE ACROSS ALL PERTURBATIONS")
print("="*95)

# Master Progress Bar
pbar_pert = tqdm(all_perturbations, desc="Master Evaluation Suite")

for p_type in pbar_pert:
    pbar_pert.set_postfix({"Current": p_type.upper()})
    
    tqdm.write("\n\n" + "★"*95)
    tqdm.write(f"  EVALUATING PERTURBATION: [{p_type.upper()}]")
    tqdm.write("★"*95)

    critical_thresholds = {}

    # --- A. Run Module 3: Adaptive Sampling ---
    plt.figure(figsize=(9, 4.5))
    for m_name, m_inst in trained_models.items():
        sampler = AdaptiveSamplingEngine(model=m_inst, dataloader=raw_test_loader, engine=pert_engine, device=device)
        intensities, accuracies, crit_thresh = sampler.run_adaptive_sampling(
            p_type=p_type, max_evals=9, max_batches=15
        )
        critical_thresholds[m_name] = crit_thresh
        plt.plot(intensities, accuracies, marker='o', label=f"{m_name} (Crit Thresh: {crit_thresh:.2f})")

    plt.title(f"Module 3: Adaptive Elbow Curves ({p_type.replace('_', ' ').title()})", fontweight='bold')
    plt.xlabel("Intensity (I)")
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 100)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- B. Run Module 4: Statistical Metrics ---
    tqdm.write(f"\n📊 MODULE 4: STATISTICAL METRICS AT CRITICAL THRESHOLDS ({p_type.upper()})")
    tqdm.write("-" * 95)
    tqdm.write(f"{'Model':<14} | {'Thresh':<8} | {'Accuracy':<10} | {'MCC':<8} | {'Macro F1':<10} | {'Flip Rate':<11} | {'ECE':<8} | {'NLL':<8}")
    tqdm.write("-" * 95)

    for m_name, m_inst in trained_models.items():
        thresh = critical_thresholds[m_name]
        metrics = metric_engine.evaluate_comprehensive_metrics(
            model=m_inst, dataloader=raw_test_loader, engine=pert_engine,
            p_type=p_type, intensity=thresh, device=device, max_batches=20
        )
        tqdm.write(f"{m_name:<14} | {thresh:<8.2f} | {metrics['Accuracy (%)']:<10.2f} | {metrics['MCC']:<8.3f} | {metrics['Macro F1']:<10.3f} | {metrics['Flip Rate (%)']:<11.2f} | {metrics['ECE']:<8.4f} | {metrics['NLL']:<8.4f}")
    tqdm.write("-" * 95)

    # --- C. Run Module 5: Dynamic LIME Hunter ---
    tqdm.write(f"\n🔍 MODULE 5: LIME QUALITATIVE FOCUS SHIFT ({p_type.upper()})")
    
    def find_flip_case(m_inst, p_type, thresh, max_search=100):
        m_inst.eval()
        for idx in range(min(max_search, len(raw_test_dataset))):
            raw_img, true_lbl = raw_test_dataset[idx]
            with torch.no_grad():
                clean_tensor = pert_engine.normalize(raw_img.unsqueeze(0)).to(device)
                if m_inst(clean_tensor).argmax(dim=1).item() != true_lbl: 
                    continue 
                
                pert_tensor = pert_engine.apply_perturbation(raw_img.unsqueeze(0), p_type, thresh).to(device)
                pert_pred = m_inst(pert_tensor).argmax(dim=1).item()
                
                if pert_pred != true_lbl:
                    return raw_img, true_lbl, pert_pred
        return raw_test_dataset[0][0], raw_test_dataset[0][1], -1

    for m_name, m_inst in trained_models.items():
        crit_thresh = critical_thresholds[m_name]
        fail_img, true_lbl, wrong_pred = find_flip_case(m_inst, p_type, crit_thresh)
        
        if wrong_pred != -1:
            tqdm.write(f">>> Explaining [{m_name}] failure at I={crit_thresh:.2f} (Flipped to: {classes[wrong_pred]})")
        else:
            tqdm.write(f">>> Explaining [{m_name}] at I={crit_thresh:.2f} (Fallback image used)")
            
        lime_engine = LIMEExplainabilityEngine(model=m_inst, engine=pert_engine, classes=classes, device=device)
        lime_engine.explain_critical_sample(
            raw_img_tensor=fail_img, label_idx=true_lbl, p_type=p_type, 
            critical_intensity=crit_thresh, num_samples=150
        )

In [ ]:
# 1. Select a clean sample from raw test dataset
sample_idx = 12  
raw_sample_img, true_label = raw_test_dataset[sample_idx]

# Classes are dynamically inherited from Snippet 8
all_perturbations = [
    "gaussian_blur", "gaussian_noise", "brightness", "rotation", 
    "contrast", "sensor_interference", "environmental_obstruction", "capture_instability"
]

eval_intensity = 0.40  # Meaningful stress-test intensity

print("\n" + "="*90)
print(f"🖼️ RENDERING COMPARATIVE PERTURBATION SAMPLES (True Class: {classes[true_label].upper()})")
print("="*90)

for m_name, model in trained_models.items():
    print(f"\n>>> Generating Matrix for Architecture: [{m_name}]")
    model.eval()
    
    fig, axes = plt.subplots(len(all_perturbations), 3, figsize=(10, 2.8 * len(all_perturbations)))
    
    with torch.no_grad():
        clean_tensor = pert_engine.normalize(raw_sample_img.unsqueeze(0)).to(device)
        clean_probs = F.softmax(model(clean_tensor), dim=1).cpu().numpy()[0]
        clean_pred = np.argmax(clean_probs)
        clean_conf = clean_probs[clean_pred] * 100.0

    for idx, p_type in enumerate(all_perturbations):
        with torch.no_grad():
            raw_pert = pert_engine.apply_raw_perturbation(raw_sample_img.unsqueeze(0), p_type, eval_intensity).squeeze(0)
            norm_pert = pert_engine.normalize(raw_pert.unsqueeze(0)).to(device)
            pert_probs = F.softmax(model(norm_pert), dim=1).cpu().numpy()[0]
            pert_pred = np.argmax(pert_probs)
            pert_conf = pert_probs[pert_pred] * 100.0
            
        clean_display = raw_sample_img.permute(1, 2, 0).numpy()
        pert_display = raw_pert.permute(1, 2, 0).numpy()

        axes[idx, 0].imshow(clean_display)
        axes[idx, 0].set_title(f"Clean Input\nPred: {classes[clean_pred]} ({clean_conf:.1f}%)", fontsize=9)
        axes[idx, 0].axis('off')

        axes[idx, 1].imshow(pert_display)
        is_correct = (pert_pred == true_label)
        status_color = "green" if is_correct else "red"
        axes[idx, 1].set_title(f"[{p_type.replace('_', ' ').title()}]\nPred: {classes[pert_pred]} ({pert_conf:.1f}%)", 
                               fontsize=9, color=status_color, fontweight='bold')
        axes[idx, 1].axis('off')

        top3_indices = np.argsort(pert_probs)[-3:][::-1]
        top3_labels = [classes[i] for i in top3_indices]
        top3_vals = [pert_probs[i] * 100.0 for i in top3_indices]
        
        axes[idx, 2].barh(top3_labels, top3_vals, color='skyblue')
        axes[idx, 2].set_xlim(0, 100)
        axes[idx, 2].set_xlabel("Confidence (%)", fontsize=8)
        axes[idx, 2].invert_yaxis()
        axes[idx, 2].tick_params(axis='both', which='major', labelsize=8)

    plt.suptitle(f"Model Diagnostic Matrix: {m_name} (Intensity I = {eval_intensity})", fontsize=13, fontweight='bold', y=1.002)
    plt.tight_layout()
    plt.show()